# Avance 4 · Regresión de la potencia en la fase 2

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este notebook mide la potencia de una molécula (pActividad) en lugar de predecir si está
activa o no. El objetivo es ordenar los candidatos, para quedarse con los que más potencia
tienen sobre una proteína.


## 1. Preparación

In [ ]:
import os
# Un solo hilo por modelo, como en el notebook 04: así la columna de tiempos es comparable.
os.environ["OMP_NUM_THREADS"] = "1"

import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.metrics import mean_absolute_error, matthews_corrcoef, roc_auc_score, root_mean_squared_error
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn):
    print(f"{m.__name__:<8} {m.__version__}")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESADOS = RAIZ / "data" / "processed"

sistema  Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.41
python   3.12.15
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1


### 2. Los datos

La unidad es el par compuesto-diana. Se usan los mismos pares de entrenamiento y validación
que la clasificación del notebook 04, y la partición de prueba no se toca. La diana entra como
columnas de 0 y 1, porque sin ella el modelo no sabe contra qué se midió.

De esos pares, solo los que tienen potencia exacta sirven para entrenar una regresión sencilla.
Los que solo traen cotas se cargan igual, en `limite_inferior` y `limite_superior`, porque la
concordancia de Harrell sí puede usarlos al evaluar aunque no se entrene con ellos.


In [ ]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc2 = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = (pares[pares["activo_10um"].notna() & pares["particion"].ne("prueba")]   # la prueba no se toca
     .merge(desc2, on="smiles").reset_index(drop=True))
cols = [c for c in desc2.columns if c != "smiles"]
# La diana entra como columnas 0/1: sin ella el modelo no sabe contra qué se midió.
X = np.hstack([d[cols].to_numpy("float32"), pd.get_dummies(d["target_gene"]).to_numpy("float32")])
y_pot = d["pActividad"].to_numpy()                  # la potencia; NaN si el par solo tiene cotas
y_cls = d["activo_10um"].astype(int).to_numpy()     # la etiqueta, para comparar con la clasificación
g = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()               # solo_chembl, solo_npass, mixto
lim_inf = d["limite_inferior"].to_numpy()           # cotas del intervalo, para la concordancia
lim_sup = d["limite_superior"].to_numpy()
exacta = ~np.isnan(y_pot)
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta · {(~exacta).sum():,} solo con cotas")

54,369 pares · 46,092 con potencia exacta · 8,277 solo con cotas


## 3. Seis modelos de regresión, más un piso

| Clasificación (notebook 04) | Regresión (aquí) |
|:--|:--|
| logística | `Ridge` (regresión lineal) |
| Naive Bayes | `BayesianRidge` (regresión bayesiana) |
| árbol | `DecisionTreeRegressor` |
| kNN | `KNeighborsRegressor` |
| SVM-RBF | `SVR` |
| MLP | `MLPRegressor` |

`media` aparece en la tabla de resultados pero **no es uno de los seis**, es el piso: predice
siempre el promedio, así que es la marca de «no aprender nada». Un modelo que no le gane no
aprendió nada útil.

**Esto es un piso optimista, no un resultado.** Los seis se entrenan solo con las mediciones
exactas, y eso deja fuera la mitad de los negativos: de los 15,554 pares inactivos, 7,536
—el 48.5%— existen únicamente como cotas, un «> 10 µM» sin número. Son justo los compuestos
flojos, así que el modelo aprende de un mundo más fácil que el real. Quien use la censura es
el baseline del 03b, y la comparación entre los dos es la que aísla cuánto cuesta tirarlos.

Cada modelo se evalúa con la misma validación cruzada agrupada del notebook 04 (5 pliegues por
`grupo_cv`, semillas 0, 1 y 2, que también fijan el azar de cada modelo).


In [ ]:
MODELOS = {
    "media":  DummyRegressor(),                                          # el piso: no aprende nada
    "ridge":  make_pipeline(MinMaxScaler(), Ridge()),
    "bayes_ridge": make_pipeline(MinMaxScaler(), BayesianRidge()),
    "arbol":  DecisionTreeRegressor(min_samples_leaf=5),
    "knn":    make_pipeline(MinMaxScaler(), KNeighborsRegressor(n_neighbors=15)),
    "svr":    make_pipeline(MinMaxScaler(), SVR()),
    "mlp":    make_pipeline(MinMaxScaler(), MLPRegressor(hidden_layer_sizes=(64,), max_iter=600)),
}
SEMILLAS = (0, 1, 2)
CORTE = 5.0   # pActividad 5 = 10 µM, el mismo corte de la clasificación


def medir(y_pot, y_cls, exacta, pred):
    """Error y orden sobre las exactas; MCC y AUROC sobre todos los pares, cortando en 5."""
    return {"mae": mean_absolute_error(y_pot[exacta], pred[exacta]),
            "rmse": root_mean_squared_error(y_pot[exacta], pred[exacta]),
            "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
            "mcc": matthews_corrcoef(y_cls, pred >= CORTE),
            "auroc": roc_auc_score(y_cls, pred)}


filas = []
for semilla in SEMILLAS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=semilla)
    for k, (ajuste, val) in enumerate(cv.split(X, y_cls, g), 1):
        ajuste_ex = ajuste[exacta[ajuste]]           # se entrena solo con las exactas
        for nombre, modelo in MODELOS.items():
            m = clone(modelo)
            # La semilla fija también el azar del modelo (árbol, MLP), no solo el reparto.
            m.set_params(**{p: semilla for p in m.get_params() if p.endswith("random_state")})
            reloj = time.perf_counter()
            m.fit(X[ajuste_ex], y_pot[ajuste_ex])
            seg = time.perf_counter() - reloj
            pred = m.predict(X[val])
            fila = {"modelo": nombre, "semilla": semilla, "pliegue": k, "seg": seg}
            filas.append({**fila, "estrato": "todos",
                          **medir(y_pot[val], y_cls[val], exacta[val], pred)})
            for e in np.unique(procedencia[val]):
                sel = procedencia[val] == e
                ys = y_cls[val][sel]
                if min(ys.sum(), (1 - ys).sum()) >= 20:      # estrato con 20 casos de cada clase
                    filas.append({**fila, "estrato": e,
                                  **medir(y_pot[val][sel], ys, exacta[val][sel], pred[sel])})
res = pd.DataFrame(filas)

/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,


/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,


/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,


/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,


/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,


...

/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,


/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,


/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,


/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,
/tmp/ipykernel_9/623877881.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "spearman": spearmanr(y_pot[exacta], pred[exacta]).statistic,


/usr/local/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (600) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
def tabla(res, estrato="todos"):
    """Media por modelo, ordenada por Spearman. `desv_mcc` es dispersión entre pliegues, no error estándar."""
    r = res[res["estrato"] == estrato]
    t = r.groupby("modelo")[["mae", "rmse", "spearman", "mcc", "auroc", "seg"]].mean()
    t.insert(3, "desv_mcc", r.groupby("modelo")["mcc"].std())
    return t.sort_values("spearman", ascending=False).round(3)


tabla_todos = tabla(res)
tabla_todos

,mae,spearman,mcc,desv_mcc,auroc,seg
modelo,,,,,,
knn,0.769,0.658,0.343,0.015,0.803,0.054
mlp,0.823,0.642,0.372,0.022,0.801,51.121
svr,0.794,0.641,0.350,0.016,0.818,141.753
ridge,0.907,0.529,0.220,0.025,0.774,0.109
arbol,0.986,0.484,0.291,0.021,0.717,2.888
media,1.100,NaN,0.000,0.000,0.500,0.017


### 4. Por procedencia

Un promedio puede esconder cómo le va a los naturales, que son los que este proyecto existe
para priorizar. En clasificación ya se había visto: en el Avance 3 el bosque bajaba de 0.858
de AUROC en los pares de solo ChEMBL a 0.611 en los de solo NPASS. Aquí se revisa si la
regresión repite esa caída.


In [ ]:
por_procedencia = {e: tabla(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
pd.concat(por_procedencia, names=["procedencia"])[["spearman", "mcc", "auroc", "rmse"]]

spearman    mcc  auroc
procedencia modelo                        
solo_chembl knn        0.642  0.320  0.796
            mlp        0.629  0.346  0.795
            svr        0.622  0.313  0.808
            ridge      0.502  0.168  0.761
            arbol      0.469  0.273  0.711
            media        NaN  0.000  0.500
mixto       knn        0.282  0.142  0.616
            mlp        0.249  0.168  0.623
            svr        0.228  0.135  0.604
            arbol      0.128  0.096  0.556
            ridge      0.062 -0.006  0.510
            media        NaN  0.000  0.500
solo_npass  knn        0.362 -0.002  0.492
            svr        0.313  0.018  0.542
            mlp        0.260  0.071  0.569
            ridge      0.256  0.005  0.521
            arbol      0.181 -0.001  0.510
            media        NaN  0.000  0.500

## 5. Lectura


In [6]:
# Registro en experimentos/ (decisión 2.9: clasificar con un corte o predecir la potencia).
sys.path.insert(0, str(RAIZ))          # así lo hace el notebook 03
from src.tracking import Corrida

with Corrida("baseline-regresion-sencilla-exactas", decision="2.9",
             params={"modelos": list(MODELOS), "pliegues": 5, "entrena_con": "solo exactas",
                     "objetivo": "pActividad", "corte_mcc": CORTE, "estratos": "procedencia"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": tabla_todos.to_dict(orient="index"),
                **{e: t.to_dict(orient="index") for e, t in por_procedencia.items()}})

print(c.carpeta.name)

2026-10-25-antonio-2.9-baseline-regresion-sencilla-exactas


# 6. Declaración de uso de IA y contribución
Qué hice — revisar y validar tus ediciones de código, redactar los textos de las celdas 0, 3, 5 y 8 sobre lo que tú decidiste, proponer los cuatro títulos.
Qué no hice — elegir los modelos, interpretar resultados, escribir la lectura o la declaración.
Qué validaste tú — ajustaste cada texto, elegiste BayesianRidge entre las tres opciones de Pame, y decidiste dejar media como piso.